# 第 6 节课 · U-Net 语义分割实战

## 本 Notebook 目标

完成本 Notebook 后，你将能够：
1. 理解语义分割与分类、检测任务的区别
2. 掌握 U-Net 的编码器-解码器架构 + Skip Connection 设计思想
3. 理解上采样/转置卷积的作用
4. 理解 Dice Loss 与 CrossEntropy Loss 在分割任务中的差异
5. 在 Oxford-IIIT Pet 数据集上训练一个轻量 U-Net
6. 可视化分割结果：原图 + mask 叠加

## 语义分割是什么？

计算机视觉任务可以按输出粒度分为三类：

| 任务 | 输出 | 说明 |
|------|------|------|
| **图像分类** | 1 个类别标签 | 图里有什么 |
| **目标检测** | N 个边界框 + 类别 | 目标在哪里（框级别） |
| **语义分割** | 每个像素的类别标签 | 每个像素属于什么类别 |

语义分割的输出和输入图片尺寸相同，每个像素都有一个类别预测。

应用场景：自动驾驶（道路/行人/车辆分割）、医学影像（器官/肿瘤分割）、遥感图像、人像抠图等。


## 1. 环境准备


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else
                     'mps' if torch.backends.mps.is_available() else
                     'cpu')
print(f"使用设备: {device}")


## 2. U-Net 架构详解

U-Net 是 2015 年提出的医学图像分割网络，结构像字母 "U"，因此得名。

### 核心组成

1. **编码器（Encoder / 下采样路径）**
   - 由多个卷积 + 池化层组成
   - 逐步降低空间分辨率，提取高级语义特征
   - 类似分类网络的前半部分

2. **瓶颈（Bottleneck）**
   - 编码器和解码器之间的最窄处
   - 分辨率最低，通道数最高

3. **解码器（Decoder / 上采样路径）**
   - 由多个转置卷积 + 卷积组成
   - 逐步恢复空间分辨率
   - 最终输出与输入相同尺寸的预测图

4. **Skip Connection（跳跃连接）**
   - 把编码器每一层的特征直接拼接到解码器对应层
   - 作用：保留空间细节，帮助解码器恢复精确边界
   - 这是 U-Net 最关键的设计

### 为什么需要 Skip Connection？

编码器不断下采样，会丢失细节信息。解码器如果只依靠瓶颈处的特征上采样，恢复出的边界会比较模糊。

Skip Connection 把编码器浅层的高分辨率特征直接传过来，让解码器"回忆"起细节，从而输出更精确的分割边界。


## 3. 上采样方法

分割网络需要把低分辨率特征图恢复到原始尺寸。常见上采样方法：

### 3.1 最近邻插值
直接复制最近的像素值，简单但不够平滑。

### 3.2 双线性插值
用周围像素加权平均，结果更平滑，但没有可学习参数。

### 3.3 转置卷积（ConvTranspose2d）
带可学习参数的上采样，网络可以自己学到最佳上采样方式。

U-Net 中使用的是转置卷积。


In [ ]:
# 对比三种上采样方法
x = torch.randn(1, 1, 4, 4)
print(f"输入尺寸: {x.shape}")

# 最近邻插值
up_nearest = F.interpolate(x, scale_factor=2, mode='nearest')
print(f"最近邻插值: {up_nearest.shape}")

# 双线性插值
up_bilinear = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
print(f"双线性插值: {up_bilinear.shape}")

# 转置卷积
conv_t = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2)
up_conv_t = conv_t(x)
print(f"转置卷积: {up_conv_t.shape}")


## 4. 损失函数：CrossEntropy vs Dice

### 4.1 CrossEntropy Loss
逐像素做分类。问题是：如果前景像素很少（如医学图像中的肿瘤），背景会主导损失，模型可能倾向于全部预测为背景。

### 4.2 Dice Loss
衡量预测 mask 和真实 mask 的重叠程度：

$$\text{Dice} = \frac{2 |X \cap Y|}{|X| + |Y|}$$

Dice Loss = 1 - Dice

优点：对类别不平衡更鲁棒，即使前景像素少也能有效优化。

实践中常用 **CE + Dice 组合**。


In [ ]:
def dice_loss(pred, target, smooth=1.0):
    """
    计算 Dice Loss。

    pred: [B, C, H, W] softmax 概率
    target: [B, H, W] 类别索引
    """
    num_classes = pred.size(1)
    # 把 target 转成 one-hot
    target_one_hot = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    pred = pred.softmax(dim=1)
    intersection = (pred * target_one_hot).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()


# 测试 dice_loss
pred_test = torch.randn(2, 3, 64, 64)
target_test = torch.randint(0, 3, (2, 64, 64))
loss = dice_loss(pred_test, target_test)
print(f"Dice Loss: {loss.item():.4f}")


## 5. 加载 Oxford-IIIT Pet 数据集

这是一个宠物图片分割数据集，包含 37 个类别的猫和狗。每张图片有对应的像素级 mask，mask 包含 3 类：
- 0：背景
- 1：宠物边界
- 2：宠物主体

torchvision 提供自动下载。


In [ ]:
# 图像预处理
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# mask 预处理（不需要标准化，只需要 resize）
target_transform = transforms.Compose([
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor()
])

# 加载数据集
train_dataset = datasets.OxfordIIITPet(
    './data',
    split='trainval',
    target_types='segmentation',
    download=True,
    transform=transform,
    target_transform=target_transform
)

test_dataset = datasets.OxfordIIITPet(
    './data',
    split='test',
    target_types='segmentation',
    download=True,
    transform=transform,
    target_transform=target_transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"训练集: {len(train_dataset):,} 张")
print(f"测试集: {len(test_dataset):,} 张")
print(f"类别数: 3（背景、边界、主体）")


## 6. 可视化数据集


In [ ]:
def denormalize(tensor):
    """反标准化，用于显示"""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    img, mask = train_dataset[i]
    img_display = denormalize(img).permute(1, 2, 0).numpy()
    mask_display = mask.squeeze().numpy()

    axes[0, i].imshow(img_display)
    axes[0, i].set_title('Image')
    axes[0, i].axis('off')

    axes[1, i].imshow(mask_display, cmap='jet', vmin=0, vmax=2)
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')

plt.suptitle('Oxford-IIIT Pet 数据集示例', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. 定义轻量 U-Net

我们实现一个 4 层编码器-解码器的轻量 U-Net，适合 CPU/GPU 训练。


In [ ]:
class UNet(nn.Module):
    """轻量 U-Net"""
    def __init__(self, in_ch=3, out_ch=3):
        super().__init__()
        # 编码器
        self.enc1 = self.conv_block(in_ch, 32)
        self.enc2 = self.conv_block(32, 64)
        self.enc3 = self.conv_block(64, 128)
        self.enc4 = self.conv_block(128, 256)
        self.pool = nn.MaxPool2d(2)

        # 瓶颈
        self.bottleneck = self.conv_block(256, 512)

        # 解码器
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = self.conv_block(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self.conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(64, 32)

        self.out = nn.Conv2d(32, out_ch, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # 编码器
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # 瓶颈
        b = self.bottleneck(self.pool(e4))

        # 解码器 + Skip Connection
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


# 测试模型
model = UNet(in_ch=3, out_ch=3).to(device)
x_test = torch.randn(2, 3, 128, 128).to(device)
y_test = model(x_test)
print(f"输入形状:  {x_test.shape}")
print(f"输出形状:  {y_test.shape}")
print(f"参数量:    {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


## 8. 训练函数

我们分别用 Dice Loss 和 CrossEntropy Loss 训练两个模型，对比效果。


In [ ]:
def train_unet(model, train_loader, test_loader, epochs=10, use_dice=True, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'test_loss': []}

    print(f"{'Epoch':<8} {'Train Loss':<12} {'Test Loss':<12}")
    print("-" * 35)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.squeeze(1).long().to(device)

            optimizer.zero_grad()
            outputs = model(images)

            if use_dice:
                loss = dice_loss(outputs, masks)
            else:
                loss = F.cross_entropy(outputs, masks)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # 测试
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for images, masks in test_loader:
                images, masks = images.to(device), masks.squeeze(1).long().to(device)
                outputs = model(images)
                if use_dice:
                    loss = dice_loss(outputs, masks)
                else:
                    loss = F.cross_entropy(outputs, masks)
                test_loss += loss.item()
        test_loss /= len(test_loader)

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        print(f"{epoch:<8} {train_loss:<12.4f} {test_loss:<12.4f}")

    return history


## 9. 实验一：用 Dice Loss 训练 U-Net

Dice Loss 对类别不平衡更鲁棒，在分割任务中常用。


In [ ]:
print("=" * 50)
print("实验一：Dice Loss 训练 U-Net")
print("=" * 50)
model_dice = UNet(in_ch=3, out_ch=3).to(device)
history_dice = train_unet(model_dice, train_loader, test_loader, epochs=10, use_dice=True, lr=0.001)


## 10. 实验二：用 CrossEntropy Loss 训练 U-Net


In [ ]:
print("=" * 50)
print("实验二：CrossEntropy Loss 训练 U-Net")
print("=" * 50)
model_ce = UNet(in_ch=3, out_ch=3).to(device)
history_ce = train_unet(model_ce, train_loader, test_loader, epochs=10, use_dice=False, lr=0.001)


## 11. 损失曲线对比


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = list(range(1, 11))

axes[0].plot(epochs, history_dice['train_loss'], 'o-', color='#0d9f8f', linewidth=2, label='Dice')
axes[0].plot(epochs, history_ce['train_loss'], 's-', color='#c25e00', linewidth=2, label='CrossEntropy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
axes[0].set_title('训练 Loss 对比'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_dice['test_loss'], 'o-', color='#0d9f8f', linewidth=2, label='Dice')
axes[1].plot(epochs, history_ce['test_loss'], 's-', color='#c25e00', linewidth=2, label='CrossEntropy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Loss')
axes[1].set_title('测试 Loss 对比'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 12. 可视化分割结果


In [ ]:
def visualize_predictions(model, dataset, num_samples=4):
    model.eval()
    fig, axes = plt.subplots(3, num_samples, figsize=(14, 9))

    for i in range(num_samples):
        img, mask = dataset[i]
        img_input = img.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_input)
            pred = output.argmax(dim=1).squeeze(0).cpu().numpy()

        img_display = denormalize(img).permute(1, 2, 0).numpy()
        mask_true = mask.squeeze().numpy()

        axes[0, i].imshow(img_display)
        axes[0, i].set_title('Image')
        axes[0, i].axis('off')

        axes[1, i].imshow(mask_true, cmap='jet', vmin=0, vmax=2)
        axes[1, i].set_title('Ground Truth')
        axes[1, i].axis('off')

        axes[2, i].imshow(pred, cmap='jet', vmin=0, vmax=2)
        axes[2, i].set_title('Prediction')
        axes[2, i].axis('off')

    plt.suptitle('U-Net 分割结果可视化（Dice Loss）', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


visualize_predictions(model_dice, test_dataset, num_samples=4)


## 13. 计算 mIoU

mIoU（mean Intersection over Union）是分割任务最常用的评估指标。

对于每个类别：
$$\text{IoU}_c = \frac{\text{预测为 c 且真实为 c 的像素数}}{\text{预测为 c 或真实为 c 的像素数}}$$

mIoU = 所有类别 IoU 的平均值


In [ ]:
def compute_miou(model, data_loader, num_classes=3):
    model.eval()
    intersection = torch.zeros(num_classes)
    union = torch.zeros(num_classes)

    with torch.no_grad():
        for images, masks in data_loader:
            images = images.to(device)
            masks = masks.squeeze(1).long().to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            for c in range(num_classes):
                pred_c = (preds == c)
                true_c = (masks == c)
                intersection[c] += (pred_c & true_c).sum().item()
                union[c] += (pred_c | true_c).sum().item()

    ious = intersection / (union + 1e-8)
    miou = ious.mean().item()
    return ious, miou


print("=" * 50)
print("mIoU 评估")
print("=" * 50)
ious_dice, miou_dice = compute_miou(model_dice, test_loader)
ious_ce, miou_ce = compute_miou(model_ce, test_loader)

print(f"Dice Loss 模型:")
print(f"  各类 IoU: {ious_dice.numpy()}")
print(f"  mIoU: {miou_dice:.4f}")
print()
print(f"CrossEntropy Loss 模型:")
print(f"  各类 IoU: {ious_ce.numpy()}")
print(f"  mIoU: {miou_ce:.4f}")


## 14. U-Net 中的 Skip Connection 有多重要？

Skip Connection 把编码器的高分辨率特征传给解码器。如果没有它，解码器只能依赖瓶颈处的低分辨率特征，恢复出的边界会模糊。

你可以想象：
- 编码器下采样就像把图片越看越远，只能看到整体轮廓
- Skip Connection 就像给解码器一面放大镜，让它能看到细节

这也是为什么 U-Net 在医学图像等小目标分割上效果特别好。


## 15. 动手练习

### 练习 1：增加训练轮数
把 epochs 从 10 增加到 20 或 30，观察分割效果是否继续提升。

### 练习 2：去掉 Skip Connection
修改 U-Net，去掉所有 Skip Connection（即不 concatenate 编码器特征），重新训练，观察边界模糊程度。

### 练习 3：调整学习率
尝试 lr=0.0001 和 lr=0.01，观察训练稳定性。

### 练习 4：更深的 U-Net
把 U-Net 从 4 层改成 5 层编码器-解码器，参数量和效果有什么变化？


## 16. 常见问题

**Q1：U-Net 只能做二分类分割吗？**
A：不是。U-Net 可以做任意类别数的分割，输出通道数等于类别数。

**Q2：为什么分割任务的输入输出尺寸相同？**
A：因为我们要给每个像素分类，所以输出需要保持和输入相同的空间分辨率。

**Q3：Dice Loss 和 CE Loss 哪个更好？**
A：没有绝对答案。类别不平衡严重时 Dice 通常更好；类别较均衡时 CE 也可以。实际中常组合使用。

**Q4：Skip Connection 和 ResNet 的残差连接有什么区别？**
A：ResNet 的 shortcut 是相加（$y = F(x) + x$），U-Net 的 skip 是拼接（concatenate），把两个特征图在通道维度拼起来。


## 17. 小结

- 语义分割是像素级分类，输出和输入尺寸相同
- U-Net = 编码器 + 瓶颈 + 解码器 + Skip Connection
- Skip Connection 保留细节，帮助解码器恢复精确边界
- 上采样常用转置卷积或插值
- Dice Loss 对类别不平衡更鲁棒，分割任务常用
- mIoU 是分割的主要评估指标

下一节课我们将学习 SAM：不需要训练就能做分割的零样本模型。
